# Frozen inference on RunPod
Use one RTX 4090 (24 GB), On-Demand, and an official RunPod PyTorch template with Linux, Python >= 3.11 and JupyterLab. Configure persistent `/workspace` storage. Upload the existing `nlp-colab-assets.zip` to `/workspace`; no training or new model upload is needed.

This is a separate hardware replication. Mac and Colab results remain separate. No speedup or completion cost is assumed: measure the development stage before launching the full benchmark. GPU and storage billing are controlled in your RunPod console, not by this notebook. See `docs/runpod_inference.md` for setup and backup details.

In [ ]:
from pathlib import Path, PurePosixPath
import sys, subprocess, json, zipfile, hashlib, time, os
assert sys.platform.startswith('linux'), 'Use a Linux RunPod GPU Pod'
assert sys.version_info >= (3, 11), 'Choose a template with Python 3.11 or newer'
assert Path('/workspace').is_dir(), 'Mount persistent storage at /workspace first'
subprocess.run(['nvidia-smi'], check=True)
BUNDLE = Path('/workspace/nlp-colab-assets.zip')
ROOT = Path('/workspace/nlp-runpod')
VENV = Path('/workspace/nlp-runpod-env')
STATE = Path('/workspace/nlp-runpod-state')
REPO_URL = 'https://github.com/faisalAI27/Qwen2.5-1.5B-finetune.git'
EXPECTED_BUNDLE_SHA256 = 'fbe767cf2ed584134ce822204ae46e70461b7000cc01b51bcccb84f2eba3a926'

In [ ]:
assert BUNDLE.is_file(), f'Upload the existing asset ZIP to {BUNDLE}'
hasher = hashlib.sha256()
with BUNDLE.open('rb') as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b''):
        hasher.update(chunk)
assert hasher.hexdigest() == EXPECTED_BUNDLE_SHA256, 'Unexpected asset ZIP; do not mix bundles'
with zipfile.ZipFile(BUNDLE) as archive:
    bundled = archive.read('colab_bundle_manifest.json')
    metadata = json.loads(bundled)
    if metadata['bundle_mode'] == 'assets_with_github_code':
        assert metadata['source_commit'], 'Asset bundle must pin a source commit'
        if not (ROOT / '.git').exists():
            subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
        subprocess.run(['git', 'checkout', '--detach', metadata['source_commit']], cwd=ROOT, check=True)
    ROOT.mkdir(parents=True, exist_ok=True)
    old = ROOT / 'colab_bundle_manifest.json'
    assert not old.exists() or old.read_bytes() == bundled, 'Use a fresh workspace/state folder for a different bundle'
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        assert not relative.is_absolute() and '..' not in relative.parts, 'Unsafe archive path'
    archive.extractall(ROOT)
original = ROOT / 'manifests/evaluation_freeze_macos.json'
canonical = ROOT / 'manifests/evaluation_freeze.json'
if canonical.exists() and canonical.read_bytes() == original.read_bytes():
    canonical.rename(ROOT / 'manifests/evaluation_freeze_macos_from_clone.json')
(ROOT / 'RUNPOD_ISOLATED_WORKSPACE').write_text('Independent RunPod hardware replication.\n')
os.environ['HF_HOME'] = str(ROOT / 'cache/huggingface')
os.environ['MLX_PTX_CACHE_DIR'] = str(ROOT / 'cache/mlx-ptx')

In [ ]:
RUNPOD_LAUNCHER = '"""Run the verified CUDA pipeline as a separate RunPod hardware replication."""\n\nfrom functools import lru_cache\nimport importlib.util\nfrom pathlib import Path\nimport sys\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\nfrom s1small.data import file_sha256\n\nDEV_RUN = "runpod-cuda-dev-v1"\nTEST_RUN = "math500-runpod-cuda-v1"\n\n\ndef runpod_identity(base, launcher):\n    return {**base, "provider": "runpod", "runpod_launcher_sha256": file_sha256(launcher)}\n\n\ndef main():\n    launcher = Path(__file__).resolve()\n    root = launcher.parents[1]\n    if not (root / "RUNPOD_ISOLATED_WORKSPACE").is_file():\n        raise ValueError("Use the separate exported RunPod workspace, never the active Mac or Colab run")\n    from s1small import colab_cuda\n    original_signature = colab_cuda.runtime_signature\n\n    @lru_cache(maxsize=1)\n    def signature():\n        return runpod_identity(original_signature(), launcher)\n\n    # Reuse every original CUDA/model/protocol check. The provider and launcher\n    # hash become part of smoke, development and frozen benchmark identities.\n    colab_cuda.runtime_signature = signature\n    spec = importlib.util.spec_from_file_location("runpod_verified_cuda_runner", root / "scripts/colab_inference.py")\n    runner = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(runner)\n    runner.DEV_RUN = DEV_RUN\n    runner.TEST_RUN = TEST_RUN\n    runner.main()\n\n\nif __name__ == "__main__":\n    main()\n'

launcher_path = ROOT / 'scripts/runpod_inference.py'
assert not launcher_path.exists() or launcher_path.read_text() == RUNPOD_LAUNCHER, 'Launcher differs: use a fresh workspace for changed launcher code'
launcher_path.write_text(RUNPOD_LAUNCHER)
environment = subprocess.run([sys.executable, '-m', 'venv', str(VENV)], capture_output=True, text=True)
if environment.returncode:
    print(environment.stdout, environment.stderr)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'virtualenv==20.36.1'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--no-periodic-update', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(ROOT / 'requirements-colab.txt')], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '--no-cache-dir', 'nvidia-cuda-runtime-cu12==12.9.79', 'nvidia-cuda-nvrtc-cu12==12.9.86'], check=True)
cuda_root = Path(subprocess.check_output([PYTHON, '-c', "from importlib.metadata import distribution; print(distribution('nvidia-cuda-runtime-cu12').locate_file('nvidia/cuda_runtime'))"], text=True).strip())
assert (cuda_root / 'include/cuda_runtime.h').is_file(), 'CUDA runtime headers missing'
os.environ['CUDA_HOME'] = str(cuda_root)
os.environ['CUDA_PATH'] = str(cuda_root)
print('Using local CUDA headers:', cuda_root / 'include')
subprocess.run([PYTHON, str(ROOT / 'scripts/colab_checkpoint.py'), 'restore', '--drive-folder', str(STATE)], check=True)
subprocess.run([PYTHON, str(launcher_path), 'verify'], check=True)

## GPU preflight and helpers
The preflight uses a fresh process and the project interpreter. The helper prints newly appended output only, reports GPU activity every 30 seconds, and saves mutable state every five minutes and after each stage. A same-volume snapshot does not protect against Pod deletion: download backups before terminating a Pod.

In [ ]:
subprocess.run([PYTHON, "-c", "import mlx.core as mx\nmx.set_default_device(mx.gpu)\nx = mx.arange(16).reshape(4, 4).astype(mx.bfloat16)\ny = mx.take(x, mx.array([1, 3], dtype=mx.int32), axis=0)\nmx.eval(y)\nprint('GPU gather preflight passed:', y.astype(mx.float32).tolist())\n"], check=True)

(ROOT / 'results').mkdir(exist_ok=True)
packages = subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True)
(ROOT / 'results/runpod-installed-packages.txt').write_text(packages)

def save_state():
    subprocess.run([PYTHON, str(ROOT / 'scripts/colab_checkpoint.py'), 'save', '--drive-folder', str(STATE)], check=True)

def run_stage(mode):
    log = ROOT / 'results' / ('runpod-' + mode + '.log')
    log.parent.mkdir(parents=True, exist_ok=True)
    position = log.stat().st_size if log.exists() else 0
    def show_new():
        nonlocal position
        with log.open('rb') as reader:
            reader.seek(position)
            content = reader.read()
            position = reader.tell()
        if content:
            print(content.decode(errors='replace')[-10000:], end='', flush=True)
    def gpu_sample():
        sample = subprocess.run(['nvidia-smi', '--query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,power.draw', '--format=csv,noheader'], capture_output=True, text=True)
        if sample.returncode == 0:
            print('GPU activity:', sample.stdout.strip(), flush=True)
            with (ROOT / 'results/runpod-gpu.csv').open('a') as output:
                output.write(sample.stdout)
        else:
            print('GPU monitoring unavailable:', sample.stderr[-300:], flush=True)
    with log.open('ab', buffering=0) as stream:
        process = subprocess.Popen([PYTHON, str(launcher_path), mode], cwd=ROOT, stdin=subprocess.DEVNULL, stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
        checkpoint_due = time.monotonic() + 300
        try:
            while process.poll() is None:
                time.sleep(30)
                show_new()
                if process.poll() is None:
                    gpu_sample()
                if time.monotonic() >= checkpoint_due:
                    save_state()
                    checkpoint_due = time.monotonic() + 300
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            save_state()
            raise
    show_new()
    save_state()
    if process.returncode:
        print('\n'.join(log.read_text(errors='replace').splitlines()[-100:]))
        raise RuntimeError(f'{mode} exited {process.returncode}; saved evidence: {log}')
    print(f'{mode}: completed successfully', flush=True)

## 1. Smoke
Checks the exact trained adapter, unchanged weights and a bounded development answer. It does not grade accuracy or retrain anything.

In [ ]:
run_stage('smoke')

## 2. Development speed and correctness-of-protocol checks
Run all 45 outputs on the same five held-out questions. Allow first-run JIT compilation. Compare actual timings before committing to a full paid benchmark. Output extraction is not mathematical accuracy.

In [ ]:
run_stage('development')

## 3. Inspect measurements and freeze
The next cell validates the full development coverage and records this RunPod GPU/runtime. It prints the Mac and RunPod mean times and exact-token parity; differences in generated lengths affect the time comparison. The benchmark remains a separate RunPod replication.

In [ ]:
run_stage('freeze')
run_stage('status')

## 4. Full benchmark (paid GPU time)
Run this cell after reviewing development timings and the current hourly charge. This plans 4,500 outputs under `math500-runpod-cuda-v1`. Previously saved slots from this exact RunPod runtime are retained on resume. Do not copy Mac or Colab predictions into this run.

In [ ]:
run_stage('test')

## 5. Grade, analyze and export
Grading and analysis use the pinned stack; they do not need GPU acceleration. GPU billing nevertheless continues while the Pod remains running. Download the final result archive and latest state snapshot, then stop the Pod in the console. Storage remains billed while retained; deleting the Pod can delete its volume.

In [ ]:
run_stage('grade')
run_stage('analyze')
run_stage('status')

In [ ]:
export = Path('/workspace/nlp-runpod-results.zip')
with zipfile.ZipFile(export, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ('results', 'manifests'):
        for path in (ROOT / folder).rglob('*'):
            if path.is_file():
                archive.write(path, str(path.relative_to(ROOT)))
    for path in (launcher_path, ROOT / 'colab_bundle_manifest.json'):
        archive.write(path, str(path.relative_to(ROOT)))
print('Download:', export)
print('Latest state snapshots:', sorted(STATE.glob('cuda-state-*.zip')))
print('Stop the Pod in RunPod console when finished; this notebook does not stop billing.')